In [2]:
import sys
import numpy as np
import torch

sys.path.append("../")
from datasets.dataloader import get_wireless_dataloader

def angles_to_quaternion(azimuth, elevation):
    w = np.cos(azimuth / 2.0) * np.cos(elevation / 2.0)
    x = -np.sin(azimuth / 2.0) * np.sin(elevation / 2.0)
    y = np.cos(azimuth / 2.0) * np.sin(elevation / 2.0)
    z = np.sin(azimuth / 2.0) * np.cos(elevation / 2.0)
    return np.array([w, x, y, z], dtype=np.float32)

dataloader = get_wireless_dataloader(
    "../datasets/outputs/conf_16x2_414u_5.0ghz_sbrRT_sc104.mat",
    batch_size=16,
    num_pc=16378,
    drop_last=True
)

receiver_data = []

for batch_idx, batch in enumerate(dataloader):
    point_clouds     = batch["point_cloud"]
    tx_positions     = batch["tx_position"]
    rx_positions     = batch["rx_position"]
    channel_matrices = batch["channel_matrix"]
    aoa_list         = batch["aoa"]
    env_dims         = batch["env_dims"]

    print("Shapes from the dataloader (batch index = {}):".format(batch_idx))
    if isinstance(point_clouds, torch.Tensor):
        print("  point_clouds:", point_clouds.shape)
    else:
        print("  point_clouds is a list of length:", len(point_clouds))
        print("    first item shape:", point_clouds[0].shape)
    print("  tx_positions:", tx_positions.shape)
    print("  rx_positions:", rx_positions.shape)
    print("  channel_matrices:", channel_matrices.shape)
    print("  aoa_list length:", len(aoa_list))
    print("  env_dims:", env_dims.shape)

    batch_size = rx_positions.shape[0]
    for i in range(batch_size):
        rx_pos = rx_positions[i].cpu().numpy()
        current_aoa = aoa_list[i]
        if current_aoa.ndim == 1:
            azimuth   = current_aoa[0].item()
            elevation = current_aoa[1].item()
        else:
            azimuth   = current_aoa[0, -1].item()
            elevation = current_aoa[1, -1].item()
        quaternion = angles_to_quaternion(azimuth, elevation)
        receiver_data.append({
            "rx_position": rx_pos,
            "quaternion":  quaternion
        })

print("\nTotal receiver_data entries:", len(receiver_data))
for entry in receiver_data[:5]:
    print("Receiver Position:", entry["rx_position"],
          "Quaternion:", entry["quaternion"])


Shapes from the dataloader (batch index = 0):
  point_clouds: torch.Size([16378, 3])
  tx_positions: torch.Size([3])
  rx_positions: torch.Size([16, 3])
  channel_matrices: torch.Size([16, 16, 2, 2])
  aoa_list length: 16
  env_dims: torch.Size([3, 2])
Shapes from the dataloader (batch index = 1):
  point_clouds: torch.Size([16378, 3])
  tx_positions: torch.Size([3])
  rx_positions: torch.Size([16, 3])
  channel_matrices: torch.Size([16, 16, 2, 2])
  aoa_list length: 16
  env_dims: torch.Size([3, 2])
Shapes from the dataloader (batch index = 2):
  point_clouds: torch.Size([16378, 3])
  tx_positions: torch.Size([3])
  rx_positions: torch.Size([16, 3])
  channel_matrices: torch.Size([16, 16, 2, 2])
  aoa_list length: 16
  env_dims: torch.Size([3, 2])
Shapes from the dataloader (batch index = 3):
  point_clouds: torch.Size([16378, 3])
  tx_positions: torch.Size([3])
  rx_positions: torch.Size([16, 3])
  channel_matrices: torch.Size([16, 16, 2, 2])
  aoa_list length: 16
  env_dims: torch.S